# 1. Configuración Inicial y Conexión a Base de Datos
Definimos las rutas de origen de los CSV y la cadena de conexión a SQL Server.

In [1]:
import pandas as pd
import os
import re
import pyodbc
from pathlib import Path

# Definir la fecha de corte para este proceso (formato AAAA-MM-DD)
FECHA_CORTE = "2026-10-08"  # <-- Cambia esta fecha según necesites

ORIGEN = Path(r"C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Convenios's files - OXFORD TUTORIALS\pruebas PM")

TABLA_STAGE = 'dbo.avance_oxford_stage_new'
# Configuración de Conexión a Azure SQL (Nube)
DB_SERVER = 'bi-db-group.database.windows.net'
DB_NAME = 'rendimiento_academico'
DB_USER = 'bi_admin'
DB_PASS = '53/O~D&2m#o}'

CONN_STR = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};' 
    f'SERVER={DB_SERVER};' 
    f'DATABASE={DB_NAME};' 
    f'UID={DB_USER};' 
    f'PWD={DB_PASS};' 
    'Encrypt=yes;' 
    'TrustServerCertificate=no;' 
    'Connection Timeout=30;'
)

# Mensajes de confirmación visual
print("✅ Configuración inicial y variables cargadas correctamente.")
print(f"📅 Fecha de corte definida: {FECHA_CORTE}")
print(f"📁 Ruta de origen configurada: {ORIGEN}")
print(f"☁️ Base de datos en la nube: {DB_SERVER} -> {DB_NAME}")

# Buscar y mostrar los archivos que están en la carpeta
if ORIGEN.exists():
    archivos_csv = list(ORIGEN.glob("*.csv")) # Busca archivos que terminen en .csv
    if len(archivos_csv) > 0:
        print(f"\n📄 Se encontraron {len(archivos_csv)} archivo(s) CSV listos para procesar:")
        for archivo in archivos_csv:
            print(f"   - {archivo.name}")
    else:
        print("\n⚠️ La carpeta existe, pero actualmente no hay ningún archivo .csv adentro.")
else:
    print("\n❌ La carpeta de origen NO existe en tu computador. Por favor revisa la ruta.")


✅ Configuración inicial y variables cargadas correctamente.
📅 Fecha de corte definida: 2026-10-08
📁 Ruta de origen configurada: C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Convenios's files - OXFORD TUTORIALS\pruebas PM
☁️ Base de datos en la nube: bi-db-group.database.windows.net -> rendimiento_academico

📄 Se encontraron 131 archivo(s) CSV listos para procesar:
   - 2026-10-19_1039_B1.1-5PM87_SDDE_2026-2_B1.1-5PM87 SDDE_2026-2_19Oct2026.csv
   - 2026-10-19_1039_B1.1-7PM147_SDDE_2026-2_B1.1-7PM147 SDDE_2026-2_19Oct2026.csv
   - 2026-10-19_1040_B1.1-7PM149_SDDE_2026-2_B1.1-7PM149 SDDE_2026-2_19Oct2026.csv
   - 2026-10-19_1041_B1.1-5PM91_SDDE_2026-2_B1.1-5PM91 SDDE_2026-2_19Oct2026.csv
   - 2026-10-19_1041_B1.1-7PM151_SDDE_2026-2_B1.1-7PM151 SDDE_2026-2_19Oct2026.csv
   - 2026-10-19_1042_B1.1-5PM92_SDDE_2026-2_B1.1-5PM92 SDDE_2026-2_19Oct2026.csv
   - 2026-10-19_1043_B1.1-5PM93_SDDE_2026-2_B1.1-5PM93 SDDE_2026-2_19Oct2026.csv
   - 2026-10-19_1043_B1.1-7PM152_SDDE_2026-

In [2]:
PATRON_CURSO = re.compile(r"""(?ix)
    (?P<clase>[a-z]{1,3}\s*\d+(?:[.\-_]\d+)?)    
    [\s\-_]*
    (?P<grupo>\d{1,2}\s*[ap]\.?m\.?)             
    [\s\-_]*
    (?P<codigo>\d+)                              
    (?:[\s\-_]+(?P<segmento>[a-z]{2,10}))?       
""")
MESES = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06",
         "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
def partes_del_nombre(nombre_archivo):
    nombre = Path(nombre_archivo).stem
    nombre = re.sub(r"^\d{4}-\d{2}-\d{2}_\d{3,4}_", "", nombre)
    hallado = PATRON_CURSO.search(nombre)
    if not hallado: return {"clase": None, "codigo": None, "grupo": None, "segmento": None}
    return {
        "clase": re.sub(r"[\s_\-]", ".", hallado["clase"].strip()).upper(),
        "codigo": hallado["codigo"],
        "grupo": re.sub(r"[\s.]", "", hallado["grupo"]).upper(),
        "segmento": (hallado["segmento"] or "").upper() or None,
    }
def a_porcentaje(val):
    val = str(val).strip(' "')
    if val in ("", "Never"): return None
    if val == "-": return "-"
    
    val_clean = val.replace("%", "").strip()
    
    # Si es una fraccion como "20/283", la devolvemos tal cual como texto
    if "/" in val_clean:
        return val_clean
        
    try:
        return float(val_clean)
    except ValueError:
        return val

def a_fecha(val):
    val = str(val).strip(' \"')
    if val in ("", "-", "Never"): return None
    try:
        dia, mes, anio = val.split()
        return f"{anio}-{MESES[mes]}-{dia.zfill(2)}"
    except Exception: return None

In [3]:
def validar_estructura(row1, nombre_archivo):
    if len(row1) < 54: pass
    if not row1[4].strip(' \"').startswith("Total"):
        raise ValueError("La columna índice 4 no es 'Total'.")
    categorias_dinamicas = []
    num_blocks = (len(row1) - 6) // 4
    for i in range(num_blocks):
        base = 6 + i * 4
        val_base = row1[base].strip(' \"')
        val_base_norm = " ".join(val_base.split())
        if not val_base_norm: break
        match = re.match(r"(?i)^Unit\s+(\d+)\s+(.+)$", val_base_norm)
        if match:
            categorias_dinamicas.append({'unit': int(match.group(1)), 'category': match.group(2).strip().lower().replace(" ", "_")})
        else:
            raise ValueError(f"Error formato columna: '{val_base}'")
    return categorias_dinamicas

In [4]:
def crear_consolidado(origen_dir, fecha_corte):
    archivos = list(Path(origen_dir).glob("*.csv"))
    lista_dfs = []
    errores_locales = []
    print("⏳ Extrayendo datos de los archivos CSV...")
    
    for archivo in archivos:
        try:
            with open(archivo, "r", encoding="utf-16", errors="replace") as f:
                lineas = [l.rstrip('\r\n') for l in f if l.strip()]
                if len(lineas) < 4: raise ValueError("Archivo muy corto.")
                
                rows = [line.split('\t') for line in lineas]
                categorias_dinamicas = validar_estructura(rows[0], archivo.name)
                
                nombres_columnas = ['teacher', 'learning_material', 'student', 'last_opened', 'clase', 'codigo', 'grupo', 'segmento', 'fecha_corte', 'total_completed_pct', 'total_score_pct']
                for cat in categorias_dinamicas:
                    u = f"u{cat['unit']:02d}"
                    c = cat['category']
                    nombres_columnas.extend([f"{u}_{c}_completed_pct", f"{u}_{c}_score_pct", f"{u}_{c}_test_completed_pct", f"{u}_{c}_test_score_pct"])
                
                teacher_val = rows[2][0].strip(' \"') if len(rows[2]) > 0 else None
                material_val = rows[2][1].strip(' \"') if len(rows[2]) > 1 else None
                meta = partes_del_nombre(archivo.name)
                datos_archivo = []
                
                for row in rows[3:]:
                    if not any(c.strip() for c in row): continue
                    req_len = 6 + (len(categorias_dinamicas) * 4)
                    if len(row) < req_len: row.extend([""] * (req_len - len(row)))
                    
                    # AQUI SE ASIGNA EL PARÁMETRO fecha_corte AL DATAFRAME
                    param_row = [
                        teacher_val, material_val, row[2].strip(' \"'), a_fecha(row[3]),
                        meta["clase"], meta["codigo"], meta["grupo"], meta["segmento"], fecha_corte,
                        a_porcentaje(row[4]), a_porcentaje(row[5])
                    ]
                    for i in range(len(categorias_dinamicas)):
                        base = 6 + i * 4
                        param_row.extend([a_porcentaje(row[base]), a_porcentaje(row[base+1]), a_porcentaje(row[base+2]), a_porcentaje(row[base+3])])
                    datos_archivo.append(param_row)
                
                if datos_archivo:
                    df = pd.DataFrame(datos_archivo, columns=nombres_columnas)
                    df['archivo_origen'] = archivo.name # Útil para rastrear errores
                    lista_dfs.append(df)
        except Exception as e:
            errores_locales.append(f"{archivo.name} -> {e}")
            
    if errores_locales:
        print("\n⚠️ HUBO ERRORES AL LEER ESTOS ARCHIVOS (Fueron saltados):")
        for e in errores_locales: print(f" - {e}")
            
    return pd.concat(lista_dfs, ignore_index=True) if lista_dfs else pd.DataFrame()


In [5]:
df_excel = crear_consolidado(ORIGEN, FECHA_CORTE)

if not df_excel.empty:
    print(f"\n✅ ¡Éxito! Se consolidaron {len(df_excel)} estudiantes.")
    display(df_excel) # Muestra a los estudiantes como tabla
else:
    print("\n❌ No se encontraron datos para consolidar.")


⏳ Extrayendo datos de los archivos CSV...

✅ ¡Éxito! Se consolidaron 4147 estudiantes.


,teacher,learning_material,student,last_opened,clase,codigo,grupo,segmento,fecha_corte,total_completed_pct,...,u10_theories_test_completed_pct,u10_theories_test_score_pct,u11_lifestyle_completed_pct,u11_lifestyle_score_pct,u11_lifestyle_test_completed_pct,u11_lifestyle_test_score_pct,u12_character_completed_pct,u12_character_score_pct,u12_character_test_completed_pct,u12_character_test_score_pct
0,DIANA CAMPOS NARVAEZ,Wide Angle Level 3 Online Practice,Maria 1108558398,2026-10-05,B1.1,87,5PM,SDDE,2026-10-08,6/293,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DIANA CAMPOS NARVAEZ,Wide Angle Level 3 Online Practice,Dulce 4995742,NaN,B1.1,87,5PM,SDDE,2026-10-08,0/293,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DIANA CAMPOS NARVAEZ,Wide Angle Level 3 Online Practice,Tatiana 1014482177,2026-10-13,B1.1,87,5PM,SDDE,2026-10-08,107/293,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DIANA CAMPOS NARVAEZ,Wide Angle Level 3 Online Practice,Linda 1018475022,NaN,B1.1,87,5PM,SDDE,2026-10-08,0/293,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DIANA CAMPOS NARVAEZ,Wide Angle Level 3 Online Practice,Alexander 79584231,2026-10-12,B1.1,87,5PM,SDDE,2026-10-08,35/293,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4142,Diego Pacheco,Wide Angle Level 5 Online Practice,ANDRES 1023976176,NaN,C1.1,217,7PM,SDDE,2026-10-08,0/286,...,0/3,-,0/21,-,0/3,-,0/21,-,0/3,-
4143,Diego Pacheco,Wide Angle Level 5 Online Practice,Nicolas 1019844064,NaN,C1.1,217,7PM,SDDE,2026-10-08,0/286,...,0/3,-,0/21,-,0/3,-,0/21,-,0/3,-
4144,Diego Pacheco,Wide Angle Level 5 Online Practice,Miguel 1025322163,NaN,C1.1,217,7PM,SDDE,2026-10-08,0/286,...,0/3,-,0/21,-,0/3,-,0/21,-,0/3,-
4145,Diego Pacheco,Wide Angle Level 5 Online Practice,Andrés 1011320167,NaN,C1.1,217,7PM,SDDE,2026-10-08,0/286,...,0/3,-,0/21,-,0/3,-,0/21,-,0/3,-


In [6]:
df_109 = df_excel[df_excel['codigo'] == '109']
df_109
#df_109 = df_109[['u01_achievements_completed_pct']]
#df_109

,teacher,learning_material,student,last_opened,clase,codigo,grupo,segmento,fecha_corte,total_completed_pct,...,u10_theories_test_completed_pct,u10_theories_test_score_pct,u11_lifestyle_completed_pct,u11_lifestyle_score_pct,u11_lifestyle_test_completed_pct,u11_lifestyle_test_score_pct,u12_character_completed_pct,u12_character_score_pct,u12_character_test_completed_pct,u12_character_test_score_pct
1702,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Andrés 1028487302,NaN,B1.2,109,5PM,SDDE,2026-10-08,0/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1703,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Yessica 1032398900,2026-10-14,B1.2,109,5PM,SDDE,2026-10-08,37/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1704,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Nidia Gil Abril,2026-10-18,B1.2,109,5PM,SDDE,2026-10-08,45/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1705,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Diana 53039261,2026-10-15,B1.2,109,5PM,SDDE,2026-10-08,56/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1706,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Santiago 1028485645,2026-10-18,B1.2,109,5PM,SDDE,2026-10-08,56/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1707,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Beatriz 1024522921,2026-10-14,B1.2,109,5PM,SDDE,2026-10-08,44/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1708,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,YES Andres Riaño 1028487302,2026-10-14,B1.2,109,5PM,SDDE,2026-10-08,42/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1709,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Lady 51844656,NaN,B1.2,109,5PM,SDDE,2026-10-08,0/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1710,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Brenda 1073711638,2026-10-06,B1.2,109,5PM,SDDE,2026-10-08,34/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1711,Carolina Jojoa Pujimuy,Wide Angle Level 4 Online Practice,Sebastian 1000383076,NaN,B1.2,109,5PM,SDDE,2026-10-08,0/283,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
import pyodbc
import pandas as pd
import numpy as np
from datetime import datetime

def subir_directo_a_tablas_finales(df, fecha_corte):
    if df.empty:
        print("❌ El DataFrame está vacío. No hay nada que subir.")
        return
        
    print("⏳ Limpiando datos en memoria...")
    df_limpio = df.dropna(subset=['student', 'learning_material']).copy()
    
    # Limpiar espacios en blanco y filtrar filas inválidas
    df_limpio['student'] = df_limpio['student'].astype(str).str.strip()
    df_limpio['learning_material'] = df_limpio['learning_material'].astype(str).str.strip()
    df_limpio = df_limpio[df_limpio['student'] != ""]
    df_limpio = df_limpio[df_limpio['learning_material'] != ""]
    df_limpio = df_limpio[df_limpio['student'].str.lower() != 'class average']
    
    # Deduplicar de forma insensible a mayúsculas/minúsculas (como SQL Server)
    # dando prioridad al registro con actividad más reciente ('last_opened')
    df_limpio['_student_key'] = df_limpio['student'].str.upper()
    df_limpio['_material_key'] = df_limpio['learning_material'].str.upper()
    df_limpio = df_limpio.sort_values(by='last_opened', ascending=False, na_position='last')
    df_limpio = df_limpio.drop_duplicates(subset=['_student_key', '_material_key'], keep='first')
    df_limpio = df_limpio.drop(columns=['_student_key', '_material_key'])
    
    columnas_df = [c for c in df_limpio.columns if c != 'archivo_origen']
    df_limpio = df_limpio[columnas_df]
    
    df_limpio['created_at'] = datetime.now()
    df_limpio['updated_at'] = datetime.now()
    columnas_df.extend(['created_at', 'updated_at'])
    
    conn = pyodbc.connect(CONN_STR)
    cursor = conn.cursor()
    cursor.fast_executemany = True
    
    TABLA_HISTORICO = 'dbo.avance_oxford_historico_new'
    TABLA_OPERATIVA = 'dbo.avance_oxford_operativo_new'
    
    print("⚙️ Creando columnas dinámicas en SQL (si faltan)...")
    for tabla in [TABLA_HISTORICO, TABLA_OPERATIVA]:
        for col in columnas_df:
            if col.startswith('u') and col.endswith('_pct'):
                alter_sql = f"""
                IF NOT EXISTS (
                    SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('{tabla}') AND name = '{col}'
                )
                BEGIN
                    ALTER TABLE {tabla} ADD {col} VARCHAR(50);
                END
                """
                cursor.execute(alter_sql)
    conn.commit()
    
    print(f"🧹 Borrando cortes antiguos ({fecha_corte}) para evitar duplicados...")
    cursor.execute(f"DELETE FROM {TABLA_HISTORICO} WHERE fecha_corte = '{fecha_corte}'")
    conn.commit()
    
    print(f"🚀 Subiendo {len(df_limpio)} estudiantes validados al HISTÓRICO...")
    cols_str = ", ".join(columnas_df)
    placeholders = ", ".join(["?"] * len(columnas_df))
    sql_insert = f"INSERT INTO {TABLA_HISTORICO} ({cols_str}) VALUES ({placeholders})"
    
    df_para_sql = df_limpio.where(pd.notnull(df_limpio), None)
    params_lote_crudos = df_para_sql.values.tolist()
    
    params_lote = []
    for fila in params_lote_crudos:
        fila_limpia = []
        for val in fila:
            if isinstance(val, str) and val.strip() == "":
                fila_limpia.append(None)
            elif pd.isna(val):
                fila_limpia.append(None)
            else:
                fila_limpia.append(val)
        params_lote.append(fila_limpia)
    
    cursor.executemany(sql_insert, params_lote)
    conn.commit()
    
    print("🔄 Refrescando la tabla OPERATIVA...")
    cursor.execute(f"""
        SELECT COLUMN_NAME 
        FROM INFORMATION_SCHEMA.COLUMNS 
        WHERE TABLE_NAME = 'avance_oxford_operativo_new' 
        AND COLUMNPROPERTY(OBJECT_ID(TABLE_SCHEMA + '.' + TABLE_NAME), COLUMN_NAME, 'IsIdentity') = 0
    """)
    cols_operativo = [row.COLUMN_NAME for row in cursor.fetchall()]
    
    # ⚠️ AQUI ESTA LA MAGIA: Le agregamos "h." a cada columna para que SQL no se confunda
    cols_dinamicas_str = ", ".join(cols_operativo)
    cols_select_str = ", ".join([f"h.{col}" for col in cols_operativo])
    
    query_operativo = f"""
    TRUNCATE TABLE {TABLA_OPERATIVA};

    WITH UltimosCortes AS (
        SELECT TOP (2) fecha_corte
        FROM {TABLA_HISTORICO}
        GROUP BY fecha_corte
        ORDER BY fecha_corte DESC
    )
    INSERT INTO {TABLA_OPERATIVA} ({cols_dinamicas_str})
    SELECT {cols_select_str}
    FROM {TABLA_HISTORICO} AS h
    INNER JOIN UltimosCortes AS c ON c.fecha_corte = h.fecha_corte; 
    """
    cursor.execute(query_operativo)
    conn.commit()
    
    print("🎉 ¡Proceso finalizado con éxito!")
    
    cursor.close()
    conn.close()

# ================================
# EJECUTAR CARGA FINAL
# ================================
subir_directo_a_tablas_finales(df_excel, FECHA_CORTE)


⏳ Limpiando datos en memoria...
⚙️ Creando columnas dinámicas en SQL (si faltan)...
🧹 Borrando cortes antiguos (2026-10-08) para evitar duplicados...
🚀 Subiendo 4060 estudiantes validados al HISTÓRICO...
🔄 Refrescando la tabla OPERATIVA...
🎉 ¡Proceso finalizado con éxito!
